# Notebook 12: Text Classification — Movie Review Sentiment
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Represent text as numerical features (Bag-of-Words, TF-IDF)
- Build a text classification pipeline with scikit-learn
- Apply Naive Bayes, Logistic Regression, and LinearSVC to text
- Understand what makes text features challenging
- Interpret which words are most predictive

## 1. The Challenge of Text

ML models require fixed-length numerical vectors. Raw text is:
- **Variable length**: reviews range from 1 word to 5000+ words
- **High-dimensional**: English has 170,000+ words in common use
- **Sparse**: any single document uses only hundreds of words
- **Context-dependent**: "not bad" is positive; "good grief" is negative

### The Classic NLP Pipeline
```
Raw text
  → Tokenise (split into words)
  → Lowercase, remove punctuation
  → [Optional] Remove stop words
  → Vectorise: Bag-of-Words or TF-IDF
Numerical feature matrix  →  ML classifier
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.datasets import fetch_20newsgroups
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

## 2. Bag-of-Words (BoW)

Count how many times each word appears in a document:

```
Doc A: "I loved this movie! Great acting."
Doc B: "Terrible film. Worst movie I have seen."

Vocab: [acting, great, have, i, loved, movie, seen, terrible, this, worst]
Doc A: [  1,     1,    0,  1,    1,     1,     0,      0,       1,     0]
Doc B: [  0,     0,    1,  1,    0,     1,     1,      1,       0,     1]
```

**Problem**: common words ("the", "is", "movie") dominate and carry little meaning.

In [ ]:
corpus = [
    "I loved this movie! Fantastic acting.",
    "Terrible film. Worst movie I have seen.",
    "Great story and wonderful cinematography.",
    "Boring and predictable. Waste of time.",
    "Absolutely brilliant. A masterpiece!",
    "Not good at all. Very disappointing.",
]
labels = ['pos','neg','pos','neg','pos','neg']

vec = CountVectorizer(stop_words='english', lowercase=True)
X_bow = vec.fit_transform(corpus)
print('Vocabulary:', list(vec.vocabulary_.keys()))
print(f'Feature matrix shape: {X_bow.shape}  (6 docs × {X_bow.shape[1]} words)')
print('\nDoc 0 non-zero features:')
for idx in X_bow[0].nonzero()[1]:
    print(f'  {vec.get_feature_names_out()[idx]}: {X_bow[0,idx]}')

## 3. TF-IDF: Down-weight Common Words

$$\text{TF-IDF}(t,d) = \text{TF}(t,d) \times \log\frac{N+1}{\text{df}(t)+1}$$

- Words appearing in every document get low weight (low IDF)
- Rare but meaningful words get high weight (high IDF)
- Much better signal-to-noise ratio than raw counts

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', lowercase=True)
X_tfidf = tfidf.fit_transform(corpus)

print('TF-IDF for Doc 0:')
feat_names = tfidf.get_feature_names_out()
for idx in X_tfidf[0].nonzero()[1]:
    print(f'  {feat_names[idx]:20s}: {X_tfidf[0,idx]:.4f}')

## 4. Classification on the 20 Newsgroups Dataset

The 20 Newsgroups dataset contains ~18,000 news articles across 20 topics. We'll use 4 categories for speed.

In [ ]:
categories = ['sci.space','rec.sport.baseball','talk.politics.guns','comp.graphics']

train_ng = fetch_20newsgroups(subset='train', categories=categories,
                               remove=('headers','footers','quotes'))
test_ng  = fetch_20newsgroups(subset='test',  categories=categories,
                               remove=('headers','footers','quotes'))

print(f'Train: {len(train_ng.data)}, Test: {len(test_ng.data)}')
print(f'Categories: {train_ng.target_names}')
print(f'\nSample (truncated):\n{train_ng.data[0][:250]}...')

In [ ]:
X_tr, y_tr = train_ng.data, train_ng.target
X_te, y_te = test_ng.data,  test_ng.target

pipelines = [
    ('NB + BoW',      Pipeline([('v', CountVectorizer(stop_words='english')), ('c', MultinomialNB())])),
    ('NB + TF-IDF',   Pipeline([('v', TfidfVectorizer(stop_words='english')), ('c', MultinomialNB())])),
    ('LR + TF-IDF',   Pipeline([('v', TfidfVectorizer(stop_words='english')), ('c', LogisticRegression(max_iter=500))])),
    ('SVC + TF-IDF',  Pipeline([('v', TfidfVectorizer(stop_words='english')), ('c', LinearSVC())])),
]

print(f'{"Pipeline":<20} Test Acc')
print('-' * 35)
for name, pipe in pipelines:
    pipe.fit(X_tr, y_tr)
    print(f'{name:<20} {pipe.score(X_te, y_te):.4f}')

In [ ]:
# Confusion matrix for LinearSVC
best_pipe = pipelines[-1][1]
y_pred = best_pipe.predict(X_te)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
short = [c.split('.')[1] for c in categories]
ax.set_xticks(range(4)); ax.set_xticklabels(short, rotation=30, ha='right')
ax.set_yticks(range(4)); ax.set_yticklabels(short)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('LinearSVC — 20 Newsgroups Confusion Matrix')
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

## 5. Interpreting the Model — Which Words Matter?

In [ ]:
lr_pipe = pipelines[2][1]  # LR + TF-IDF
vect = lr_pipe.named_steps['v']
clf  = lr_pipe.named_steps['c']
feat = vect.get_feature_names_out()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, cat) in enumerate(zip(axes, categories)):
    top_idx = np.argsort(clf.coef_[i])[-10:]
    ax.barh([feat[j] for j in top_idx], clf.coef_[i][top_idx], color='steelblue')
    ax.set_title(cat.split('.')[1], fontsize=9)
    ax.tick_params(labelsize=8)
plt.suptitle('Top 10 Predictive Words per Category')
plt.tight_layout(); plt.show()

## Exercises

1. Add bigrams (`ngram_range=(1,2)`) to the TF-IDF vectorizer. Does accuracy improve?
2. Experiment with `max_features=5000` vs unlimited. What is the accuracy-speed trade-off?
3. Apply the pipeline to a binary sentiment dataset (e.g., IMDB movie reviews CSV). Can you get above 90% accuracy?
4. What are the most common confusions between `sci.space` and `comp.graphics`? Examine the misclassified documents.

## Reflection Questions
- TF-IDF treats each word independently. What language patterns does it completely miss?
- Why did we use `remove=('headers','footers','quotes')`? What is data leakage in this context?
- A LinearSVC and Logistic Regression give similar accuracy on text. Which would you choose for a production system and why?

---
**Next →** `13_model_evaluation.ipynb`